# Module 11 — RAG Evaluation Engineering
Self-contained Google Colab lab: golden data → retrieval metrics → answer metrics → judge calibration → slices → regression gate.

In [ ]:
cases=[
 {'id':'c1','query':'CICS timeout','relevant':['d1','d2'],'expected':'restart'},
 {'id':'c2','query':'AWS key','relevant':['d3'],'expected':'rotate'},
 {'id':'c3','query':'unknown policy','relevant':[],'expected':'cannot answer'}
]
def recall(ranked,relevant,k):
    return len(set(ranked[:k]) & set(relevant))/max(len(relevant),1) if relevant else 1.0
def mrr(ranked,relevant):
    for i,x in enumerate(ranked,1):
        if x in relevant:return 1/i
    return 0
print(cases)

In [ ]:
runs={
 'baseline': {'c1':['d3','d1','d2'],'c2':['d3'],'c3':[]},
 'candidate': {'c1':['d1','d2'],'c2':['d3'],'c3':[]},
}
for name,run in runs.items():
    rs=[];ms=[]
    for c in cases:
        rs.append(recall(run[c['id']],c['relevant'],2));ms.append(mrr(run[c['id']],c['relevant']))
    print(name,'Recall@2',sum(rs)/len(rs),'MRR',sum(ms)/len(ms))

## Detailed exercises
1. Expand to 50 cases.
2. Add query-class and difficulty labels.
3. Add unanswerable cases and test abstention.
4. Compute Recall@1/5/10 and MRR.
5. Add citation correctness.
6. Create human labels for 30 answers and compare an automated judge.
7. Inspect judge disagreements manually.
8. Add bootstrap confidence intervals.
9. Split metrics by query class and tenant.
10. Build a regression gate that rejects a candidate with a security regression even if accuracy improves.
11. Inject dataset leakage.
12. Demonstrate metric gaming by optimizing one metric only.

In [ ]:
def gate(baseline,candidate,max_drop=.02): return candidate >= baseline-max_drop
print('0.90 -> 0.89:',gate(.90,.89))
print('0.90 -> 0.87:',gate(.90,.87))
# Exercise: replace this single-metric gate with quality + latency + security constraints.

## Failure injection
- reward hallucinated answers for unanswerable questions
- leak evaluation answers into the corpus
- change evaluator prompt and measure judge drift
- hide a tenant regression inside the global average
- count citations without verifying support

**Gold challenge:** produce a promotion report containing metric deltas, slice regressions, confidence intervals, cost/latency impact and security findings.